In [1]:
import networkx as nx
import numpy as np
from gwgraphs import *

In [2]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("COX2_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("COX2_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("COX2_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("COX2_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("COX2_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float,
                     line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def normalize_features(features):
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()

def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels, node_attributes):
    graphs = {}
    graphs_normalized = {}
    
    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)
    
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr, norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes, normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr
        
        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix

In [3]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data(
)
graphs, graphs_normalized = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                                   node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
    )

Graph ID: 1, number of nodes: 39, label: -1
Graph ID: 2, number of nodes: 36, label: -1
Graph ID: 3, number of nodes: 36, label: 1
Graph ID: 4, number of nodes: 39, label: -1
Graph ID: 5, number of nodes: 42, label: -1
Graph ID: 6, number of nodes: 42, label: -1
Graph ID: 7, number of nodes: 44, label: -1
Graph ID: 8, number of nodes: 39, label: -1
Graph ID: 9, number of nodes: 42, label: 1
Graph ID: 10, number of nodes: 42, label: 1
Graph ID: 11, number of nodes: 45, label: 1
Graph ID: 12, number of nodes: 49, label: 1
Graph ID: 13, number of nodes: 42, label: 1
Graph ID: 14, number of nodes: 35, label: -1
Graph ID: 15, number of nodes: 44, label: 1
Graph ID: 16, number of nodes: 47, label: 1
Graph ID: 17, number of nodes: 45, label: 1
Graph ID: 18, number of nodes: 39, label: 1
Graph ID: 19, number of nodes: 44, label: -1
Graph ID: 20, number of nodes: 44, label: -1
Graph ID: 21, number of nodes: 51, label: -1
Graph ID: 22, number of nodes: 45, label: -1
Graph ID: 23, number of nodes

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_distance_matrix(graph) for graph in graphs_list]
# Determine number of elements of each class
print("Elements of Class 1: ", labels_list.count(1))
print("Elements of Class -1: ", labels_list.count(-1))

Elements of Class 1:  102
Elements of Class -1:  365


In [5]:
### fGW-kNN with COX2 data ###

In [6]:
# Balance the dataset by selecting equal numbers from each class
class_1_indices = [i for i, label in enumerate(labels_list) if label == 1]
class_neg1_indices = [i for i, label in enumerate(labels_list) if label == -1]

# Randomly select 102 elements from class -1 to match class 1
random.seed(42)  # Set seed for reproducibility
selected_neg1_indices = random.sample(class_neg1_indices, 102)

# Combine selected indices
balanced_indices = class_1_indices + selected_neg1_indices
random.shuffle(balanced_indices)  # Shuffle to mix the classes

# Create balanced dataset
balanced_graphs_list = [graphs_list[i] for i in balanced_indices]
balanced_labels_list = [labels_list[i] for i in balanced_indices]
balanced_distance_matrices = [distance_matrices[i] for i in balanced_indices]

print(f"\nBalanced dataset:")
print("Elements of Class 1: ", balanced_labels_list.count(1))
print("Elements of Class -1: ", balanced_labels_list.count(-1))

# Update the variables to use balanced data
graphs_list = balanced_graphs_list
labels_list = balanced_labels_list
distance_matrices = balanced_distance_matrices


Balanced dataset:
Elements of Class 1:  102
Elements of Class -1:  102


In [7]:
k_list = [1,2,3,4,5,6,7,8,9,10]
num_folds = 10

In [8]:
accuracies, by_class_accuracies, f1scores, best_k_info = fgw_knn(distance_matrices, graphs_list, labels_list, num_folds, k_list=k_list, method="kfold")

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 87.5%, Fold 0
k=1, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=1, Overall Classification Accuracy: 80.95238095238095%, Fold 0
k=1, F1 Score (macro avg): 80.56%, Fold 0
k=2, Class 1 Accuracy: 87.5%, Fold 0
k=2, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=2, Overall Classification Accuracy: 80.95238095238095%, Fold 0
k=2, F1 Score (macro avg): 80.56%, Fold 0
k=3, Class 1 Accuracy: 87.5%, Fold 0
k=3, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=3, Overall Classification Accuracy: 80.95238095238095%, Fold 0
k=3, F1 Score (macro avg): 80.56%, Fold 0
k=4, Class 1 Accuracy: 87.5%, Fold 0
k=4, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=4, Overall Classification Accuracy: 80.95238095238095%, Fold 0
k=4, F1 Score (macro avg): 80.56%, Fold 0
k=5, Class 1 Accuracy: 75.0%, Fold 0
k=5, Class -1 Accuracy: 69.23076923076923%, Fold 0
k=5, Overall Classification Accuracy: 71.42857142857143%, Fold 0
k=5, F1 Score (macro avg): 70.83%, Fold 

In [9]:
### Normalized feature vector experiment ###
graphs_list_norm = []
labels_list_norm = []
for graph_id, graph in graphs_normalized.items():
    # Add the graph object to the list
    graphs_list_norm.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list_norm.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices_norm = [compute_distance_matrix(graph) for graph in graphs_list_norm]

In [10]:
# Balance the dataset by selecting equal numbers from each class
class_1_indices = [i for i, label in enumerate(labels_list_norm) if label == 1]
class_neg1_indices = [i for i, label in enumerate(labels_list_norm) if label == -1]

# Randomly select 102 elements from class -1 to match class 1
random.seed(42)  # Set seed for reproducibility
selected_neg1_indices = random.sample(class_neg1_indices, 102)

# Combine selected indices
balanced_indices = class_1_indices + selected_neg1_indices
random.shuffle(balanced_indices)  # Shuffle to mix the classes

# Create balanced dataset
balanced_graphs_list = [graphs_list_norm[i] for i in balanced_indices]
balanced_labels_list = [labels_list_norm[i] for i in balanced_indices]
balanced_distance_matrices = [distance_matrices_norm[i] for i in balanced_indices]

print(f"\nBalanced dataset:")
print("Elements of Class 1: ", balanced_labels_list.count(1))
print("Elements of Class -1: ", balanced_labels_list.count(-1))

# Update the variables to use balanced data
graphs_list_norm = balanced_graphs_list
labels_list_norm = balanced_labels_list
distance_matrices_norm = balanced_distance_matrices


Balanced dataset:
Elements of Class 1:  102
Elements of Class -1:  102


In [11]:
accuracies_norm2, by_class_accuracies_norm2, f1_scores_norm2, best_k_info2 = fgw_knn(distance_matrices_norm, graphs_list_norm, labels_list_norm, num_folds, k_list=k_list, method="kfold")

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 75.0%, Fold 0
k=1, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=1, Overall Classification Accuracy: 76.19047619047619%, Fold 0
k=1, F1 Score (macro avg): 75.29%, Fold 0
k=2, Class 1 Accuracy: 75.0%, Fold 0
k=2, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=2, Overall Classification Accuracy: 76.19047619047619%, Fold 0
k=2, F1 Score (macro avg): 75.29%, Fold 0
k=3, Class 1 Accuracy: 87.5%, Fold 0
k=3, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=3, Overall Classification Accuracy: 80.95238095238095%, Fold 0
k=3, F1 Score (macro avg): 80.56%, Fold 0
k=4, Class 1 Accuracy: 75.0%, Fold 0
k=4, Class -1 Accuracy: 76.92307692307693%, Fold 0
k=4, Overall Classification Accuracy: 76.19047619047619%, Fold 0
k=4, F1 Score (macro avg): 75.29%, Fold 0
k=5, Class 1 Accuracy: 75.0%, Fold 0
k=5, Class -1 Accuracy: 69.23076923076923%, Fold 0
k=5, Overall Classification Accuracy: 71.42857142857143%, Fold 0
k=5, F1 Score (macro avg): 70.83%, Fold 